# Reto: Deserción de empleados
## The Learning Gate

-  Autor: *Pavel Vicente Alonso*
-  Fecha: *27 de marzo de 2026*


## Objetivo
Preparar un conjunto de datos adecuado para un problema de clasificación binaria de **Attrition**.

---

In [ ]:
# Importación de librerías
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA


In [ ]:
# Lectura del archivo CSV en el dataframe EmpleadosAttrition
EmpleadosAttrition = pd.read_csv('empleadosRETO.csv')
print('Dimensiones:', EmpleadosAttrition.shape)
EmpleadosAttrition.head()


Dimensiones: (400, 30)


,Age,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,NumCompaniesWorked,HiringDate,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsInCurrentRole,YearsSinceLastPromotion,Attrition
0,50,Travel_Rarely,Research & Development,1 km,2,Medical,1,997,4,Male,3,4,Research Director,4,Divorced,17399,9,06/06/2013,Y,No,22,4,3,80,32,1,2,4,1,No
1,36,Travel_Rarely,Research & Development,6 km,2,Medical,1,178,2,Male,3,2,Manufacturing Director,2,Divorced,4941,6,12/25/2015,Y,No,20,4,4,80,7,0,3,2,0,No
2,21,Travel_Rarely,Sales,7 km,1,Marketing,1,1780,2,Male,3,1,Sales Representative,2,Single,2679,1,2/14/2017,Y,No,13,3,2,80,1,3,3,0,1,Yes
3,52,Travel_Rarely,Research & Development,7 km,4,Life Sciences,1,1118,2,Male,3,3,Healthcare Representative,2,Single,10445,7,7/29/2010,Y,No,19,3,4,80,18,4,3,6,4,No
4,33,Travel_Rarely,Research & Development,15 km,1,Medical,1,582,2,Male,3,3,Manager,3,Married,13610,7,10/07/2011,Y,Yes,12,3,4,80,15,2,4,6,7,Yes


In [ ]:
# Eliminación de columnas irrelevantes
EmpleadosAttrition = EmpleadosAttrition.drop(
    columns=['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
)
EmpleadosAttrition.columns.tolist()


['Age',
 'BusinessTravel',
 'Department',
 'DistanceFromHome',
 'Education',
 'EducationField',
 'EnvironmentSatisfaction',
 'Gender',
 'JobInvolvement',
 'JobLevel',
 'JobRole',
 'JobSatisfaction',
 'MaritalStatus',
 'MonthlyIncome',
 'NumCompaniesWorked',
 'HiringDate',
 'OverTime',
 'PercentSalaryHike',
 'PerformanceRating',
 'RelationshipSatisfaction',
 'TotalWorkingYears',
 'TrainingTimesLastYear',
 'WorkLifeBalance',
 'YearsInCurrentRole',
 'YearsSinceLastPromotion',
 'Attrition']

## Creación de nuevas características

Se generan las variables:
- `Year`: año de contratación extraído de `HiringDate`
- `YearsAtCompany`: años en la empresa hasta 2018

También se limpia la variable de distancia desde casa.


In [ ]:
# Crear Year y YearsAtCompany a partir de HiringDate
# Se extrae el año directamente del texto para evitar problemas con fechas inválidas
EmpleadosAttrition['Year'] = (
    EmpleadosAttrition['HiringDate']
    .astype(str)
    .str.extract(r'(\d{4})$')
    .astype(int)
)

EmpleadosAttrition['YearsAtCompany'] = 2018 - EmpleadosAttrition['Year']

# Renombrar DistanceFromHome y crear una versión entera
EmpleadosAttrition = EmpleadosAttrition.rename(
    columns={'DistanceFromHome': 'DistanceFromHome_km'}
)

EmpleadosAttrition['DistanceFromHome'] = (
    EmpleadosAttrition['DistanceFromHome_km']
    .astype(str)
    .str.extract(r'(\d+)')
    .astype(int)
)

# Eliminar columnas temporales que ya no son útiles
EmpleadosAttrition = EmpleadosAttrition.drop(
    columns=['Year', 'HiringDate', 'DistanceFromHome_km']
)

EmpleadosAttrition.head()


,Age,BusinessTravel,Department,Education,EducationField,EnvironmentSatisfaction,Gender,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,NumCompaniesWorked,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsInCurrentRole,YearsSinceLastPromotion,Attrition,YearsAtCompany,DistanceFromHome
0,50,Travel_Rarely,Research & Development,2,Medical,4,Male,3,4,Research Director,4,Divorced,17399,9,No,22,4,3,32,1,2,4,1,No,5,1
1,36,Travel_Rarely,Research & Development,2,Medical,2,Male,3,2,Manufacturing Director,2,Divorced,4941,6,No,20,4,4,7,0,3,2,0,No,3,6
2,21,Travel_Rarely,Sales,1,Marketing,2,Male,3,1,Sales Representative,2,Single,2679,1,No,13,3,2,1,3,3,0,1,Yes,1,7
3,52,Travel_Rarely,Research & Development,4,Life Sciences,2,Male,3,3,Healthcare Representative,2,Single,10445,7,No,19,3,4,18,4,3,6,4,No,8,7
4,33,Travel_Rarely,Research & Development,1,Medical,2,Male,3,3,Manager,3,Married,13610,7,Yes,12,3,4,15,2,4,6,7,Yes,7,15


In [ ]:
# Frame informativo: sueldo promedio por departamento
SueldoPromedioDepto = (
    EmpleadosAttrition
    .groupby('Department', dropna=False)['MonthlyIncome']
    .mean()
    .reset_index(name='SueldoPromedio')
)

SueldoPromedioDepto


,Department,SueldoPromedio
0,Human Resources,6239.888889
1,Research & Development,6804.149813
2,Sales,7188.250000


## Escalamiento y codificación

- Se escala `MonthlyIncome` a un rango entre 0 y 1.
- Se convierten a numéricas las variables categóricas solicitadas.


In [ ]:
# Escalar MonthlyIncome entre 0 y 1
scaler = MinMaxScaler()
EmpleadosAttrition[['MonthlyIncome']] = scaler.fit_transform(
    EmpleadosAttrition[['MonthlyIncome']]
)

# Variables categóricas a transformar
columnas_categoricas = [
    'BusinessTravel', 'Department', 'EducationField',
    'Gender', 'JobRole', 'MaritalStatus', 'Attrition'
]

# Imputar faltantes con la moda y codificar con LabelEncoder
encoders = {}
for col in columnas_categoricas:
    if EmpleadosAttrition[col].isna().sum() > 0:
        EmpleadosAttrition[col] = EmpleadosAttrition[col].fillna(
            EmpleadosAttrition[col].mode()[0]
        )
    le = LabelEncoder()
    EmpleadosAttrition[col] = le.fit_transform(EmpleadosAttrition[col].astype(str))
    encoders[col] = dict(zip(le.classes_, le.transform(le.classes_)))

encoders


{'BusinessTravel': {'Non-Travel': np.int64(0),
  'Travel_Frequently': np.int64(1),
  'Travel_Rarely': np.int64(2)},
 'Department': {'Human Resources': np.int64(0),
  'Research & Development': np.int64(1),
  'Sales': np.int64(2)},
 'EducationField': {'Human Resources': np.int64(0),
  'Life Sciences': np.int64(1),
  'Marketing': np.int64(2),
  'Medical': np.int64(3),
  'Other': np.int64(4),
  'Technical Degree': np.int64(5)},
 'Gender': {'Female': np.int64(0), 'Male': np.int64(1)},
 'JobRole': {'Healthcare Representative': np.int64(0),
  'Human Resources': np.int64(1),
  'Laboratory Technician': np.int64(2),
  'Manager': np.int64(3),
  'Manufacturing Director': np.int64(4),
  'Research Director': np.int64(5),
  'Research Scientist': np.int64(6),
  'Sales Executive': np.int64(7),
  'Sales Representative': np.int64(8)},
 'MaritalStatus': {'Divorced': np.int64(0),
  'Married': np.int64(1),
  'Single': np.int64(2)},
 'Attrition': {'No': np.int64(0), 'Yes': np.int64(1)}}

## Evaluación estadística y selección de características

Se calcula la correlación de todas las variables con respecto a `Attrition`.

**Criterio de selección usado:** magnitud de correlación `>= 0.1`.  
Esto permite conservar tanto relaciones positivas como negativas, ya que ambas son informativas para el problema.


In [ ]:
# Correlación de cada variable con Attrition
correlacion_attrition = EmpleadosAttrition.corr(numeric_only=True)['Attrition'].sort_values(
    key=lambda s: s.abs(), ascending=False
)

correlacion_attrition


Attrition                   1.000000
JobLevel                   -0.214266
TotalWorkingYears          -0.213329
Age                        -0.212121
YearsInCurrentRole         -0.203918
MonthlyIncome              -0.194936
MaritalStatus               0.192430
YearsAtCompany             -0.176001
JobInvolvement             -0.166785
JobSatisfaction            -0.164957
EnvironmentSatisfaction    -0.124327
JobRole                     0.078684
TrainingTimesLastYear      -0.070884
YearsSinceLastPromotion    -0.069000
BusinessTravel              0.068650
PercentSalaryHike          -0.060880
Education                  -0.055531
Department                  0.054236
DistanceFromHome            0.052732
EducationField              0.051184
RelationshipSatisfaction   -0.030945
Gender                     -0.028839
WorkLifeBalance            -0.021723
NumCompaniesWorked         -0.009082
PerformanceRating          -0.006471
Name: Attrition, dtype: float64

In [ ]:
# Selección de variables con |correlación| >= 0.1
variables_seleccionadas = correlacion_attrition[
    correlacion_attrition.abs() >= 0.1
].index.tolist()

# Asegurar que Attrition permanezca como variable de salida
if 'Attrition' not in variables_seleccionadas:
    variables_seleccionadas.append('Attrition')

EmpleadosAttritionFinal = EmpleadosAttrition[variables_seleccionadas].copy()

print('Variables seleccionadas:')
print(variables_seleccionadas)
print('\nDimensiones de EmpleadosAttritionFinal:', EmpleadosAttritionFinal.shape)

EmpleadosAttritionFinal.head()


Variables seleccionadas:
['Attrition', 'JobLevel', 'TotalWorkingYears', 'Age', 'YearsInCurrentRole', 'MonthlyIncome', 'MaritalStatus', 'YearsAtCompany', 'JobInvolvement', 'JobSatisfaction', 'EnvironmentSatisfaction']

Dimensiones de EmpleadosAttritionFinal: (400, 11)


,Attrition,JobLevel,TotalWorkingYears,Age,YearsInCurrentRole,MonthlyIncome,MaritalStatus,YearsAtCompany,JobInvolvement,JobSatisfaction,EnvironmentSatisfaction
0,0,4,32,50,4,0.864269,0,5,3,4,4
1,0,2,7,36,2,0.207340,0,3,3,2,2
2,1,1,1,21,0,0.088062,2,1,3,2,2
3,0,3,18,52,6,0.497574,2,8,3,2,2
4,1,3,15,33,6,0.664470,1,7,3,3,2


## PCA

Se generan los componentes principales del frame `EmpleadosAttritionFinal` y se agregan el número mínimo de componentes que explican al menos el **80% de la varianza**.


In [ ]:
# PCA sobre EmpleadosAttritionFinal
pca = PCA()
EmpleadosAttritionPCA = pca.fit_transform(EmpleadosAttritionFinal)

varianza_acumulada = np.cumsum(pca.explained_variance_ratio_)
num_componentes = np.argmax(varianza_acumulada >= 0.80) + 1

print('Varianza acumulada:', varianza_acumulada)
print('Número mínimo de componentes para explicar al menos 80%:', num_componentes)


Varianza acumulada: [0.63586917 0.87798352 0.95703866 0.97810697 0.9846281  0.9910853
 0.99398442 0.99680918 0.9993929  0.99996943 1.        ]
Número mínimo de componentes para explicar al menos 80%: 2


In [ ]:
# Agregar componentes principales al dataframe final
for i in range(num_componentes):
    EmpleadosAttritionFinal = EmpleadosAttritionFinal.assign(
        **{f'C{i}': EmpleadosAttritionPCA[:, i]}
    )

EmpleadosAttritionFinal.head()


,Attrition,JobLevel,TotalWorkingYears,Age,YearsInCurrentRole,MonthlyIncome,MaritalStatus,YearsAtCompany,JobInvolvement,JobSatisfaction,EnvironmentSatisfaction,C0,C1
0,0,4,32,50,4,0.864269,0,5,3,4,4,20.248748,-4.208673
1,0,2,7,36,2,0.207340,0,3,3,2,2,-6.379883,-3.688133
2,1,1,1,21,0,0.088062,2,1,3,2,2,-21.512334,1.741233
3,0,3,18,52,6,0.497574,2,8,3,2,2,13.822776,-5.843291
4,1,3,15,33,6,0.664470,1,7,3,3,2,-1.435233,4.143193


In [ ]:
# Guardar archivo final
EmpleadosAttritionFinal.to_csv('EmpleadosAttritionFinal.csv', index=False)
print('Archivo guardado correctamente: EmpleadosAttritionFinal.csv')


Archivo guardado correctamente: EmpleadosAttritionFinal.csv


## Resultado final

Se generó:
- `EmpleadosAttritionFinal.csv`
- el dataframe final `EmpleadosAttritionFinal`
- el arreglo de PCA `EmpleadosAttritionPCA`
